# 07 Least Used Stations and Routes

 > Build low-usage datasets for app-side map visualization (notebook exports only).

 > Scope: raw trips in range 1-6000 per city/year.

 > Exports:
- least_used_stations.csv
- least_used_routes.csv
- least_used_start_stations.csv
- least_used_end_stations.csv
- least_used_routes_bin_summary.csv

In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from utils import load_app_ready, export_df

## Load App-Aligned Dataset

In [2]:
df = load_app_ready()
df.shape

(15907082, 22)

In [3]:
MIN_TRIPS = 1
MAX_TRIPS = 6000

required = [
    'city_name', 'year', 'trip_id',
    'start_station_name', 'start_lat', 'start_lon',
    'end_station_name', 'end_lat', 'end_lon',
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns for least-used analysis: {missing}')

base = df.dropna(
    subset=['city_name', 'year', 'trip_id', 'start_station_name', 'end_station_name', 'start_lat', 'start_lon', 'end_lat', 'end_lon']
).copy()
base['year'] = pd.to_numeric(base['year'], errors='coerce')
base = base.dropna(subset=['year']).copy()
base['year'] = base['year'].astype(int)

# Keep only distinct corridors for route-level outputs.
base_routes = base[
    base['start_station_name'].astype(str).str.strip().str.lower()
    != base['end_station_name'].astype(str).str.strip().str.lower()
]

base.shape, base_routes.shape

((15906099, 22), (15471429, 22))

In [4]:
route_year = (
    base_routes.groupby(
        [
            'city_name', 'year',
            'start_station_name', 'start_lat', 'start_lon',
            'end_station_name', 'end_lat', 'end_lon',
        ],
        as_index=False,
    )
    .agg(trips=('trip_id', 'count'))
)

route_year = route_year[route_year['trips'].between(MIN_TRIPS, MAX_TRIPS)].copy()

trip_bin_edges = [0, 500, 1000, 2000, 3000, 4000, 5000, 6000]
trip_bin_labels = [
    '1-500',
    '501-1000',
    '1001-2000',
    '2001-3000',
    '3001-4000',
    '4001-5000',
    '5001-6000',
]
route_year['trip_bin'] = pd.cut(
    route_year['trips'],
    bins=trip_bin_edges,
    labels=trip_bin_labels,
    include_lowest=True,
    right=True,
).astype(str)

route_year = route_year.sort_values(['city_name', 'year', 'trips']).reset_index(drop=True)
route_year['least_rank_city_year'] = route_year.groupby(['city_name', 'year']).cumcount() + 1

start_station_year = (
    base.groupby(['city_name', 'year', 'start_station_name', 'start_lat', 'start_lon'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .rename(columns={'start_station_name': 'station_name', 'start_lat': 'latitude', 'start_lon': 'longitude'})
)

end_station_year = (
    base.groupby(['city_name', 'year', 'end_station_name', 'end_lat', 'end_lon'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .rename(columns={'end_station_name': 'station_name', 'end_lat': 'latitude', 'end_lon': 'longitude'})
)

start_station_year = start_station_year[start_station_year['trips'].between(MIN_TRIPS, MAX_TRIPS)].copy()
end_station_year = end_station_year[end_station_year['trips'].between(MIN_TRIPS, MAX_TRIPS)].copy()

start_station_year['trip_bin'] = pd.cut(
    start_station_year['trips'],
    bins=trip_bin_edges,
    labels=trip_bin_labels,
    include_lowest=True,
    right=True,
).astype(str)
end_station_year['trip_bin'] = pd.cut(
    end_station_year['trips'],
    bins=trip_bin_edges,
    labels=trip_bin_labels,
    include_lowest=True,
    right=True,
).astype(str)

least_stations_year = (
    start_station_year.rename(columns={'trips': 'departures'})[['city_name', 'year', 'station_name', 'latitude', 'longitude', 'departures']]
    .merge(
        end_station_year.rename(columns={'trips': 'arrivals'})[['city_name', 'year', 'station_name', 'latitude', 'longitude', 'arrivals']],
        on=['city_name', 'year', 'station_name', 'latitude', 'longitude'],
        how='outer',
    )
)

for col in ['departures', 'arrivals']:
    least_stations_year[col] = least_stations_year[col].fillna(0)

least_stations_year['departures'] = least_stations_year['departures'].astype(int)
least_stations_year['arrivals'] = least_stations_year['arrivals'].astype(int)
least_stations_year['total_trips'] = least_stations_year['departures'] + least_stations_year['arrivals']

least_stations_year = least_stations_year[
    least_stations_year['total_trips'].between(MIN_TRIPS, MAX_TRIPS)
]

least_stations_year['trip_bin'] = pd.cut(
    least_stations_year['total_trips'],
    bins=trip_bin_edges,
    labels=trip_bin_labels,
    include_lowest=True,
    right=True,
).astype(str)

least_stations_year = least_stations_year.sort_values(
    ['city_name', 'year', 'total_trips'],
    ascending=[True, True, True],
).reset_index(drop=True)
least_stations_year['least_rank_city_year'] = (
    least_stations_year.groupby(['city_name', 'year']).cumcount() + 1
)

routes_bin_summary = (
    route_year.groupby(['city_name', 'year', 'trip_bin'], as_index=False)
    .agg(
        routes=('trips', 'count'),
        raw_trips=('trips', 'sum'),
    )
    .sort_values(['city_name', 'year', 'trip_bin'])
)

least_stations_year.shape, route_year.shape, start_station_year.shape, end_station_year.shape, routes_bin_summary.shape

((1980, 10), (544142, 11), (2713, 7), (2723, 7), (67, 5))

In [5]:
export_df('least_used_stations', least_stations_year)
export_df('least_used_routes', route_year)
export_df('least_used_start_stations', start_station_year)
export_df('least_used_end_stations', end_station_year)
export_df('least_used_routes_bin_summary', routes_bin_summary)

print('Exported least-used station and route datasets (raw counts).')

[bridge] Exported DataFrame: least_used_stations.csv (target: NOTEBOOK_EXPORTS_PATH)
[bridge] Exported DataFrame: least_used_routes.csv (target: NOTEBOOK_EXPORTS_PATH)
[bridge] Exported DataFrame: least_used_start_stations.csv (target: NOTEBOOK_EXPORTS_PATH)
[bridge] Exported DataFrame: least_used_end_stations.csv (target: NOTEBOOK_EXPORTS_PATH)
[bridge] Exported DataFrame: least_used_routes_bin_summary.csv (target: NOTEBOOK_EXPORTS_PATH)
Exported least-used station and route datasets (raw counts).
